# Data setup

Load the combined CRMLS sold data, clean the price columns, and test whether agent/office categorical features are associated with sale performance.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats


In [ ]:
DATA_FILE = Path("CRMLSSold_combined.csv")
PRICE_OUTPUT_FILE = Path("CRMLSSold_price_preprocessed.csv")
SIGNIFICANCE_OUTPUT_FILE = Path("agent_office_feature_significance.csv")

df = pd.read_csv(DATA_FILE, dtype=str, keep_default_na=False, low_memory=False)
df.shape


For the first pass, keep rows where the core price fields are numeric. `ClosePrice` is the result/target; `OriginalListPrice` and `ListPrice` are the price signals available before closing.

In [ ]:
price_cols = ["OriginalListPrice", "ListPrice", "ClosePrice"]

price_data = df.copy()
for col in price_cols:
    price_data[col] = pd.to_numeric(price_data[col], errors="coerce")

price_data = price_data.dropna(subset=price_cols).copy()
price_data[price_cols].to_csv(PRICE_OUTPUT_FILE, index=False)

print(f"Rows before preprocessing: {len(df):,}")
print(f"Rows after dropping missing/non-numeric prices: {len(price_data):,}")
print(f"Rows dropped: {len(df) - len(price_data):,}")


## Agent and office feature significance

Use `log(ClosePrice / ListPrice)` as the target so the test measures whether a feature is associated with selling above or below list price. With this many rows, p-values can be tiny even for weak effects, so `eta_squared` is the more useful column. Rough guide: `< 0.01` is very small, `0.01-0.06` is small, `0.06-0.14` is medium, and `> 0.14` is large.

In [ ]:
agent_features = [
    "ListAgentFirstName", "ListAgentLastName", "ListAgentFullName", "ListAgentEmail", "ListAgentAOR",
    "CoListAgentFirstName", "CoListAgentLastName", "BuyerAgentFirstName", "BuyerAgentLastName",
    "BuyerAgentMlsId", "BuyerAgentAOR", "CoBuyerAgentFirstName", "ListOfficeName", "CoListOfficeName",
    "BuyerOfficeName", "BuyerOfficeAOR",
]

agent_offices_df = price_data[agent_features + price_cols].copy()
agent_offices_df.head()


In [ ]:
model_df = agent_offices_df[
    (agent_offices_df["ListPrice"] > 0) & (agent_offices_df["ClosePrice"] > 0)
].copy()

model_df["sale_to_list_ratio"] = model_df["ClosePrice"] / model_df["ListPrice"]
model_df["log_sale_to_list_ratio"] = np.log(model_df["sale_to_list_ratio"])

min_group_size = 30
target = model_df["log_sale_to_list_ratio"].to_numpy()
grand_mean = target.mean()
ss_total = ((target - grand_mean) ** 2).sum()
results = []

for feature in agent_features:
    values = model_df[feature].fillna("").astype(str).str.strip()
    non_missing = values.ne("")
    counts = values[non_missing].value_counts()
    valid_levels = counts[counts >= min_group_size].index
    mask = non_missing & values.isin(valid_levels)

    row = {
        "feature": feature,
        "non_missing_rows": int(non_missing.sum()),
        "tested_rows": int(mask.sum()),
        "levels_total": int(counts.size),
        "levels_tested_min_30_rows": int(len(valid_levels)),
        "anova_p_value": np.nan,
        "eta_squared": np.nan,
        "mean_abs_group_effect_pct_points": np.nan,
        "top_level_by_count": counts.index[0] if len(counts) else "",
        "top_level_count": int(counts.iloc[0]) if len(counts) else 0,
    }

    if row["levels_tested_min_30_rows"] >= 2 and row["tested_rows"] > row["levels_tested_min_30_rows"]:
        temp = pd.DataFrame({
            "feature": values[mask].to_numpy(),
            "target": model_df.loc[mask, "log_sale_to_list_ratio"].to_numpy(),
        })
        group_means = temp.groupby("feature")["target"].mean()
        group_counts = temp.groupby("feature")["target"].size()
        ss_between = float((group_counts * (group_means - grand_mean) ** 2).sum())
        groups = [group["target"].to_numpy() for _, group in temp.groupby("feature")]
        _, p_value = stats.f_oneway(*groups)

        row["anova_p_value"] = float(p_value)
        row["eta_squared"] = float(ss_between / ss_total) if ss_total else np.nan
        row["mean_abs_group_effect_pct_points"] = float(np.expm1((group_means - grand_mean).abs()).mean() * 100)

    results.append(row)

significance = pd.DataFrame(results).sort_values(
    ["eta_squared", "tested_rows"], ascending=[False, False]
)
significance.to_csv(SIGNIFICANCE_OUTPUT_FILE, index=False)
significance


Interpretation from the current run: office and AOR fields are the strongest agent/office signals, but the effects are still small. Individual agent identifiers are often statistically significant too, mostly because the dataset is large, but they cover fewer rows after filtering to categories with at least 30 sales and are more likely to overfit.